In [19]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [20]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [21]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [22]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=1000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [23]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradient(model=model)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.18257060647010803
epoch:  1, loss: 0.11675401777029037
epoch:  2, loss: 0.08048482984304428
epoch:  3, loss: 0.06059003621339798
epoch:  4, loss: 0.049549926072359085
epoch:  5, loss: 0.04332762584090233
epoch:  6, loss: 0.03981667011976242
epoch:  7, loss: 0.03783745318651199
epoch:  8, loss: 0.0367228165268898
epoch:  9, loss: 0.03609568625688553
epoch:  10, loss: 0.03574150800704956
epoch:  11, loss: 0.03554028272628784
epoch:  12, loss: 0.03542417287826538
epoch:  13, loss: 0.03535565733909607
epoch:  14, loss: 0.035313043743371964
epoch:  15, loss: 0.035285212099552155
epoch:  16, loss: 0.035233449190855026
epoch:  17, loss: 0.03520134836435318
epoch:  18, loss: 0.03517269715666771
epoch:  19, loss: 0.03513272479176521
epoch:  20, loss: 0.035122886300086975
epoch:  21, loss: 0.03506986051797867
epoch:  22, loss: 0.03503730148077011
epoch:  23, loss: 0.03500619903206825
epoch:  24, loss: 0.034963756799697876
epoch:  25, loss: 0.03494548425078392
epoch:  26, loss:

In [24]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.772720217704773
Test metrics:  R2 = 0.7858226299285889
